# Lab02 — Triagem profunda

Notebook da entrega do Marco 1 (Lab02). As respostas e a leitura dos
resultados estão em `lab2_README.md`; a ficha de arquitetura, em
`lab2_ARQUITETURA.md`.

**Como executar.** As fotos do cliente não fazem parte do repositório. O
notebook lê a pasta de dados e a pasta de trabalho indicadas no `.env` da raiz
deste repositório (modelo em `.env.example`), e só reexecuta na máquina que
tem esses dados. As saídas versionadas mostram apenas contagens, métricas e
gráficos em SVG; nenhuma foto, nome de arquivo ou caminho é exibido.

**Seções:** 1. Configuração · 2. Concordância entre rotuladores (C3) ·
3. Partição sem vazamento (C4)

## 1. Configuração

In [1]:
# Configuração: versões e impressões digitais dos artefatos de entrada.
# Nenhum caminho é exibido (decisão E15): só versões e SHA-256.
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import torchvision

from meter_reader.config import get_data_dir, get_work_dir
from training.build_manifest import MANIFEST_NAME, file_sha256
from training.build_sample import SAMPLE_NAME

# Rótulos versionados neste repositório (o notebook roda a partir de lab-02/)
LABELS_PATH = Path("rotulos/lab2_rotulos.csv")
assert LABELS_PATH.is_file(), "Execute o notebook a partir da pasta lab-02/."

WORK_DIR = get_work_dir()   # valida o .env; não exibe o caminho
DATA_DIR = get_data_dir()

print(f"Python       {sys.version.split()[0]}")
print(f"PyTorch      {torch.__version__} | torchvision {torchvision.__version__}")
print(f"scikit-learn {sklearn.__version__} | numpy {np.__version__} | pandas {pd.__version__}")
print(f"GPU (CUDA)   {'sim' if torch.cuda.is_available() else 'não'}")
print()
# Impressões digitais: ligam os resultados aos mesmos dados da ET2
print(f"SHA-256 manifesto: {file_sha256(WORK_DIR / MANIFEST_NAME)}")
print(f"SHA-256 amostra:   {file_sha256(WORK_DIR / SAMPLE_NAME)}")
print(f"SHA-256 rótulos:   {file_sha256(LABELS_PATH)}")

Python       3.11.4
PyTorch      2.14.1+cu126 | torchvision 0.29.1+cu126
scikit-learn 1.9.1 | numpy 2.4.6 | pandas 3.0.6
GPU (CUDA)   sim

SHA-256 manifesto: 56467c02e64f41b6c495987e4cec4c0216dc1e38e5644ae65bceca8d2199b28a
SHA-256 amostra:   a6ced4666242f3c5832d5109c1fdc9289bbc754a693a9252971ed5259c417796
SHA-256 rótulos:   b901c05c2b5e6b89fba53adf512681a8d2507ef1aa9c2597033b6f84069c2f2e


## 2. Concordância entre rotuladores (C3)

Kappa de Cohen nas 50 fotos rotuladas às cegas pelo R1 e pelo R2, com os
rótulos brutos, antes da adjudicação. Por classe: um contra o resto. A
legibilidade é comparada só nas fotos que os dois rotularam como medidor.
Intervalo de 95% por bootstrap percentil sobre as fotos (2.000 reamostragens).
Critério da rubrica: kappa abaixo de 0,6 indica definição ambígua.

In [3]:
# C3: kappa de Cohen nas 50 fotos às cegas, com os rótulos brutos do R1 e do R2
import warnings

from sklearn.metrics import cohen_kappa_score

raw = pd.read_csv(LABELS_PATH, dtype=str, keep_default_na=False)
r2 = raw[raw["rotulador"] == "R2"].set_index("nome_arquivo")
r1 = raw[raw["rotulador"] == "R1"].set_index("nome_arquivo").loc[r2.index]  # mesmas fotos
assert len(r2) == 50 and r2.index.is_unique


def scene4(df: pd.DataFrame) -> np.ndarray:
    """Cena em 4 classes: o tipo para medidores e 'outros' para o resto."""
    return np.where(df["cena_n1"] == "outros", "outros", df["cena_n2"]).astype(str)


def kappa(y1: np.ndarray, y2: np.ndarray) -> float:
    """Kappa de Cohen; indefinido (nan) se só uma classe aparece nos dois."""
    if len(set(y1) | set(y2)) < 2:
        return float("nan")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # avisos de divisão por zero viram nan
        return float(cohen_kappa_score(y1, y2))


def bootstrap_ci(y1, y2, n_boot=2000, seed=0):
    """IC de 95% percentil; descarta reamostragens com kappa indefinido."""
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(y1), size=(n_boot, len(y1)))
    values = np.array([kappa(y1[i], y2[i]) for i in idx])
    valid = values[~np.isnan(values)]
    low, high = np.percentile(valid, [2.5, 97.5]) if len(valid) else (np.nan, np.nan)
    return low, high, len(valid)


s1, s2 = scene4(r1), scene4(r2)
both_meter = (r1["cena_n1"] == "medidor").to_numpy() & (r2["cena_n1"] == "medidor").to_numpy()

variables = {
    "Medidor × outros": (r1["cena_n1"].to_numpy(), r2["cena_n1"].to_numpy()),
    "Cena (4 classes)": (s1, s2),
    **{f"Cena: {c} × resto": (s1 == c, s2 == c)
       for c in ("digital", "ciclometrico", "indeterminado", "outros")},
    "Legibilidade (medidores)": (r1["legibilidade"].to_numpy()[both_meter],
                                 r2["legibilidade"].to_numpy()[both_meter]),
}

rows = []
for name, (y1, y2) in variables.items():
    y1, y2 = np.asarray(y1), np.asarray(y2)
    k = kappa(y1, y2)
    low, high, n_valid = bootstrap_ci(y1, y2)
    rows.append({
        "Variável": name,
        "Fotos": len(y1),
        # Casos da classe marcados por cada rotulador (só nas linhas um-contra-o-resto)
        "Casos R1/R2": f"{int(y1.sum())}/{int(y2.sum())}" if y1.dtype == bool else "—",
        "Concordância (%)": round(100 * float(np.mean(y1 == y2)), 1),
        "Kappa": round(k, 3),
        "IC 95%": f"[{low:.2f}; {high:.2f}]",
        "Reamostragens válidas": n_valid,
        "Kappa ≥ 0,6": "sim" if k >= 0.6 else ("—" if np.isnan(k) else "NÃO"),
    })

c3 = pd.DataFrame(rows).set_index("Variável")
c3

,Fotos,Casos R1/R2,Concordância (%),Kappa,IC 95%,Reamostragens válidas,"Kappa ≥ 0,6"
Variável,,,,,,,
Medidor × outros,50,—,100.0,1.000,[1.00; 1.00],1958,sim
Cena (4 classes),50,—,98.0,0.947,[0.80; 1.00],2000,sim
Cena: digital × resto,50,38/39,98.0,0.944,[0.79; 1.00],2000,sim
Cena: ciclometrico × resto,50,8/7,98.0,0.922,[0.70; 1.00],1999,sim
Cena: indeterminado × resto,50,0/0,100.0,NaN,[nan; nan],0,—
Cena: outros × resto,50,4/4,100.0,1.000,[1.00; 1.00],1958,sim
Legibilidade (medidores),46,—,97.8,0.911,[0.65; 1.00],2000,sim
